# M3B — Hadoop Streaming + Employee Sales SQL Practice
**Google Colab | Simplified from M3A**

Part C: Hadoop Streaming on logs. Part D: SQL CTEs and window functions on employee sales.


## Part A — Prepare Hadoop (Colab)
Install Hadoop 3.4.1 and set Java. This may take a few minutes.

In [1]:
import os, subprocess, pathlib, shutil, glob, re, sqlite3
from collections import Counter

def run(cmd):
    subprocess.run(cmd, shell=True, check=True)

VERSION = "3.4.1"
HOME = f"/content/hadoop-{VERSION}"
if not pathlib.Path(HOME).exists():
    run(f"wget -q https://archive.apache.org/dist/hadoop/common/hadoop-{VERSION}/hadoop-{VERSION}.tar.gz -O /content/hadoop.tar.gz")
    run("tar -xzf /content/hadoop.tar.gz -C /content")
java = subprocess.check_output("readlink -f /usr/bin/java", shell=True, text=True).strip()
os.environ["JAVA_HOME"] = str(pathlib.Path(java).parents[1])
os.environ["HADOOP_HOME"] = HOME
os.environ["PATH"] = f"{HOME}/bin:" + os.environ["PATH"]
STREAMING_JAR = glob.glob(f"{HOME}/share/hadoop/tools/lib/hadoop-streaming-*.jar")[0]
print("Hadoop ready:", HOME)

Hadoop ready: /content/hadoop-3.4.1


## Part B — Input: sample system logs
Use the same small dataset for Hadoop and SQL.

In [2]:
logs = [
    "2026-04-22 INFO BlockStateChange replication started successfully",
    "2026-04-22 WARN DataNode slow response from server",
    "2026-04-22 ERROR BlockStateChange replication failed",
    "2026-04-22 INFO replication completed successfully",
    "2026-04-22 WARN DataNode replication delayed",
]
input_path = pathlib.Path('/content/hadoop_logs.txt')
input_path.write_text('\n'.join(logs) + '\n')
print(input_path.read_text())

2026-04-22 INFO BlockStateChange replication started successfully
2026-04-22 WARN DataNode slow response from server
2026-04-22 ERROR BlockStateChange replication failed
2026-04-22 INFO replication completed successfully
2026-04-22 WARN DataNode replication delayed



## Part C — One reusable Hadoop Streaming job
The mapper emits `(word, 1)`; Hadoop sorts by word; the reducer sums counts. A single job produces results for all three exercises.

In [3]:
mapper = pathlib.Path('/content/mapper.py')
reducer = pathlib.Path('/content/reducer.py')
mapper.write_text('import sys, re\nfor line in sys.stdin:\n    for word in re.findall(r"[A-Za-z0-9_]+", line.lower()):\n        print(f"{word}\\t1")\n')
reducer.write_text('import sys\ncurrent, total = None, 0\nfor line in sys.stdin:\n    word, count = line.rstrip(\'\\n\').split(\'\\t\', 1)\n    if word != current:\n        if current is not None:\n            print(f"{current}\\t{total}")\n        current, total = word, 0\n    total += int(count)\nif current is not None:\n    print(f"{current}\\t{total}")\n')
print(mapper.read_text(), reducer.read_text())

import sys, re
for line in sys.stdin:
    for word in re.findall(r"[A-Za-z0-9_]+", line.lower()):
        print(f"{word}\t1")
 import sys
current, total = None, 0
for line in sys.stdin:
    word, count = line.rstrip('\n').split('\t', 1)
    if word != current:
        if current is not None:
            print(f"{current}\t{total}")
        current, total = word, 0
    total += int(count)
if current is not None:
    print(f"{current}\t{total}")



In [4]:
output_dir = '/content/hadoop_wordcount_output'
shutil.rmtree(output_dir, ignore_errors=True)
run(f'hadoop jar "{STREAMING_JAR}" '
    '-D mapreduce.framework.name=local -D fs.defaultFS=file:/// '
    f'-files {mapper},{reducer} '
    f'-mapper "python3 mapper.py" -reducer "python3 reducer.py" '
    f'-input {input_path} -output {output_dir}')
counts = {}
for line in pathlib.Path(output_dir, 'part-00000').read_text().splitlines():
    word, count = line.split('\t')
    counts[word] = int(count)
print('Hadoop word counts:', counts)

Hadoop word counts: {'04': 5, '2026': 5, '22': 5, 'blockstatechange': 2, 'completed': 1, 'datanode': 2, 'delayed': 1, 'error': 1, 'failed': 1, 'from': 1, 'info': 2, 'replication': 4, 'response': 1, 'server': 1, 'slow': 1, 'started': 1, 'successfully': 2, 'warn': 2}


### Exercise 1 — Word count
Which word appears most frequently?

In [5]:
print(sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:10])

[('04', 5), ('2026', 5), ('22', 5), ('replication', 4), ('blockstatechange', 2), ('datanode', 2), ('info', 2), ('successfully', 2), ('warn', 2), ('completed', 1)]


### Exercise 2 — Distinct words
The reducer emits one row per unique word.

In [6]:
print('Distinct word count:', len(counts))
print('Distinct words:', sorted(counts))

Distinct word count: 18
Distinct words: ['04', '2026', '22', 'blockstatechange', 'completed', 'datanode', 'delayed', 'error', 'failed', 'from', 'info', 'replication', 'response', 'server', 'slow', 'started', 'successfully', 'warn']


### Exercise 3 — Long words
Filter words with more than 12 characters.

In [7]:
long_words = {word: count for word, count in counts.items() if len(word) > 12}
print('Long words:', long_words)

Long words: {'blockstatechange': 2}


## Part D — SQL Practice: Employee Sales (CTE + Window Functions)
This independent SQL exercise uses **SQLite** (already available in Colab). Unlike Hadoop Streaming, it is not a distributed job.

**Goals:** compare `GROUP BY` with `SUM() OVER (PARTITION BY ...)`, calculate percentages, and rank employees.


In [8]:
import sqlite3
con = sqlite3.connect(':memory:')
con.executescript("""
CREATE TABLE sales (employee_id INTEGER, department TEXT, sale_amount INTEGER);
INSERT INTO sales VALUES
(101, 'Sales', 500), (102, 'Sales', 700), (103, 'Sales', 300),
(104, 'IT', 900), (105, 'IT', 600),
(106, 'HR', 400), (107, 'HR', 800);
""")
def show_sql(query):
    cur = con.execute(query)
    print(' | '.join(col[0] for col in cur.description))
    for row in cur.fetchall(): print(' | '.join(map(str, row)))
show_sql('SELECT * FROM sales ORDER BY department, employee_id')


employee_id | department | sale_amount
106 | HR | 400
107 | HR | 800
104 | IT | 900
105 | IT | 600
101 | Sales | 500
102 | Sales | 700
103 | Sales | 300


### SQL 1 — `GROUP BY`: one row per department
**Question:** What is the total sales amount for each department?


In [9]:
show_sql("""
SELECT department, SUM(sale_amount) AS total_sales
FROM sales
GROUP BY department
ORDER BY department;
""")


department | total_sales
HR | 1200
IT | 1500
Sales | 1500


### SQL 2 — Window function: preserve each employee row
**Question:** What is the department total alongside each employee sale?


In [10]:
show_sql("""
SELECT employee_id, department, sale_amount,
       SUM(sale_amount) OVER (PARTITION BY department) AS total_sales_per_dept
FROM sales
ORDER BY department, employee_id;
""")


employee_id | department | sale_amount | total_sales_per_dept
106 | HR | 400 | 1200
107 | HR | 800 | 1200
104 | IT | 900 | 1500
105 | IT | 600 | 1500
101 | Sales | 500 | 1500
102 | Sales | 700 | 1500
103 | Sales | 300 | 1500


### SQL 3 — CTE: percentage contribution
**Question:** What percentage of their department’s sales did each employee contribute?


In [11]:
show_sql("""
WITH dept_totals AS (
    SELECT employee_id, department, sale_amount,
           SUM(sale_amount) OVER (PARTITION BY department) AS dept_total
    FROM sales
)
SELECT employee_id, department, sale_amount, dept_total,
       ROUND(100.0 * sale_amount / dept_total, 2) AS pct_of_dept
FROM dept_totals
ORDER BY department, employee_id;
""")


employee_id | department | sale_amount | dept_total | pct_of_dept
106 | HR | 400 | 1200 | 33.33
107 | HR | 800 | 1200 | 66.67
104 | IT | 900 | 1500 | 60.0
105 | IT | 600 | 1500 | 40.0
101 | Sales | 500 | 1500 | 33.33
102 | Sales | 700 | 1500 | 46.67
103 | Sales | 300 | 1500 | 20.0


### SQL 4 — Ranking with `RANK()`
**Question:** Who sold the most in each department?


In [12]:
show_sql("""
WITH ranked AS (
    SELECT employee_id, department, sale_amount,
           RANK() OVER (PARTITION BY department ORDER BY sale_amount DESC) AS dept_rank
    FROM sales
)
SELECT * FROM ranked
WHERE dept_rank = 1
ORDER BY department;
""")


employee_id | department | sale_amount | dept_rank
107 | HR | 800 | 1
104 | IT | 900 | 1
102 | Sales | 700 | 1


### Student challenge (optional)
1. Modify SQL 2 to calculate the **average** sales per department using `AVG() OVER`.
2. Modify SQL 4 to show the **top two** employees per department.
3. Explain why `GROUP BY` returns 3 rows while the window query returns 7 rows.
4. Advanced: use `ROW_NUMBER()` instead of `RANK()` and explain how ties differ.

**Teaching comparison:** Hadoop Streaming groups key/value pairs across a shuffle; SQL `GROUP BY` aggregates groups; window functions compute group-aware values **without collapsing rows**.
